In [26]:
import cv2
import mediapipe as mp
import time 
import numpy as np
import torch 
import torch.nn as nn
import torch.nn.functional as F
import nbimporter 

# alphabet does not contain j or z because they require movement
labels = "ABCDEFGHIKLMNOPQRSTUVWXY"

# access webcam 
cap = cv2.VideoCapture(0)

In [27]:
from utils import MyConvBlock

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = torch.load( 'models/asl_model2.pth', map_location=device, weights_only=False)
model.eval()

Sequential(
  (0): MyConvBlock(
    (model): Sequential(
      (0): Conv2d(3, 25, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(25, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0, inplace=False)
      (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
  )
  (1): MyConvBlock(
    (model): Sequential(
      (0): Conv2d(25, 50, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.2, inplace=False)
      (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
  )
  (2): MyConvBlock(
    (model): Sequential(
      (0): Conv2d(50, 75, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(75, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0, 

In [28]:
class landmarker_and_result():
   def __init__(self):
      self.result = mp.tasks.vision.HandLandmarkerResult
      self.landmarker = mp.tasks.vision.HandLandmarker
      self.createLandmarker()
   
   def createLandmarker(self):
      # callback function (get result)
      def update_result(result: mp.tasks.vision.HandLandmarkerResult, output_image: mp.Image, timestamp_ms: int):
         self.result = result

      options = mp.tasks.vision.HandLandmarkerOptions( 
         base_options = mp.tasks.BaseOptions(model_asset_path="hand_landmarker.task"), # path to model
         running_mode = mp.tasks.vision.RunningMode.LIVE_STREAM, # running on a live stream
         num_hands = 2, # track both hands
         min_hand_detection_confidence = 0.3, # lower than value to get predictions more often
         min_hand_presence_confidence = 0.3, # lower than value to get predictions more often
         min_tracking_confidence = 0.3, # lower than value to get predictions more often
         result_callback=update_result) # receive the detection results asynchronously
      
      # initialize landmarker
      self.landmarker = self.landmarker.create_from_options(options)
   
   def detect_async(self, frame):
      # convert np array (vid frame) to mp image obj w/ color format
      mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
      # send to landmarker for processing 
      self.landmarker.detect_async(image = mp_image, timestamp_ms = int(time.time() * 1000))

   def close(self):
      # close landmarker
      self.landmarker.close()

In [29]:
def draw_landmarks_on_image(image, detection_result):
    try:
        # no hands detected (image not changed) 
        if not detection_result.hand_landmarks:
            return image
        
        annotated_image = image.copy()
        h, w = annotated_image.shape[:2]
        
       # hand connections defined manually
        connections = [
            [(0,1),(1,2),(2,3),(3,4)], # thumb
            [(0,5),(5,6),(6,7),(7,8)], # index
            [(0,9),(9,10),(10,11),(11,12)], # middle
            [(0,13),(13,14),(14,15),(15,16)], # ring
            [(0,17),(17,18),(18,19),(19,20)], # pinky
            [(5,9),(9,13),(13,17)] # palm
        ]

        # finger colors 
        connection_colors = [
            (0, 255, 0), # thumb
            (255, 0, 0), # index
            (0, 0, 255), # middle
            (255, 255, 0), # ring
            (0, 255, 255), # pinky
            (255, 0, 255) # palm
        ]
        
        for hand_landmarks in detection_result.hand_landmarks: # for multiple hands
            for finger_connections, color in zip(connections, connection_colors):
                # draw connections first (lines)
                # convert normalized coord -> pixels 
                for connection in finger_connections:
                    start = connection[0]
                    end = connection[1]
                    x0 = int(hand_landmarks[start].x * w)
                    y0 = int(hand_landmarks[start].y * h)
                    x1 = int(hand_landmarks[end].x * w)
                    y1 = int(hand_landmarks[end].y * h)
                    cv2.line(annotated_image, (x0, y0), (x1, y1), color, 2) 
            
            # draw landmark dots on top
            for landmark in hand_landmarks:
                x = int(landmark.x * w)
                y = int(landmark.y * h)
                cv2.circle(annotated_image, (x, y), 5, (255, 0, 0), -1)
        
        return annotated_image
    except Exception as e:
        print(f"Drawing error: {e}")
        return image

In [30]:
def get_hand_crop(frame, hand_landmarks, padding=20):
    h, w = frame.shape[:2]

    x_coord = [lm.x * w for lm in hand_landmarks]
    y_coord = [lm.y * h for lm in hand_landmarks]

    x_min = max(0, int(min(x_coord)) - padding);
    x_max = min(w, int(max(x_coord)) + padding);
    y_min = max(0, int(min(y_coord)) - padding);
    y_max = min(h, int(max(y_coord)) + padding);

    crop = frame[y_min:y_max, x_min:x_max]
    # crop = cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY)
    crop = cv2.resize(crop, (28,28))

    return crop, (x_min, y_min, x_max, y_max)


In [31]:
hand_landmarker = landmarker_and_result()

while True:
    ret, frame = cap.read()
    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    hand_landmarker.detect_async(rgb_frame)

    if hasattr(hand_landmarker.result, 'hand_landmarks') and hand_landmarker.result.hand_landmarks:
        # draw landmarks
        rgb_frame = draw_landmarks_on_image(rgb_frame, hand_landmarker.result)

        # make sure there is still at least one hand before predicting
        if len(hand_landmarker.result.hand_landmarks) > 0:
            # run prediction on first detected hand
            hand_landmarks = hand_landmarker.result.hand_landmarks[0]
            crop, (x_min, y_min, x_max, y_max) = get_hand_crop(rgb_frame, hand_landmarks)
            
            # prepare tensor for model
            tensor = torch.tensor(crop / 255.0).float() # shape: (28, 28, 3)
            tensor = tensor.permute(2, 0, 1) # shape: (3, 28, 28) 
            tensor = tensor.unsqueeze(0).to(device)  # shape: (1, 3, 28, 28)
            
            # predict
            with torch.no_grad():
                output = model(tensor)
                pred = torch.argmax(F.softmax(output, dim=1)).item()
                confidence = F.softmax(output, dim=1).max().item()
                letter = labels[pred]
        
        # display prediction on frame
        cv2.putText(rgb_frame, f"{letter} ({confidence:.0%})", 
                    (x_min, y_min - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 2)
    
    frame = cv2.cvtColor(rgb_frame, cv2.COLOR_RGB2BGR)
    
    cv2.imshow('frame', frame)

    # exit the window 
    if cv2.waitKey(1) == ord('q'):
        break
    if cv2.getWindowProperty('frame', cv2.WND_PROP_VISIBLE) < 1:
        break

In [32]:
hand_landmarker.close()
cap.release()
cv2.destroyAllWindows()